# 03 · Ablation lab: knock components out

Remove a head or an MLP and see what breaks. Two flavors:

* **mean** ablation (the default): replace the component's output by its mean over all tokens. This keeps its average contribution and removes its spatial pattern.
* **zero** ablation: remove it entirely. This is harsher and can push the model off-distribution.

Effects are on the 6-h **tendency**, in physical units.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
x = pg.state("2010-01-15T00")
a = pg.ablate(em, x, names=["L5H2"])           # also: heads=[(5, 2)], mlps=[3], mode="zero"
rc = a.rel_change()
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(rc.keys(), rc.values()); ax.set_ylabel("|Δ tendency| / |tendency|"); ax.tick_params(axis="x", rotation=90)

In [ ]:
pg.plot_maps({"Δ V_3 tendency (L5H2 ablated)": a.delta[pg.ch("V_3")],
              "Δ T_7 tendency": a.delta[pg.ch("T_7")],
              "V_3 tendency (intact)": a.base[pg.ch("V_3")],
              "Δ water_7 tendency": a.delta[pg.ch("specific_total_water_7")]}, cols=2);

## A sweep: which components matter for one quantity?
The metric here is the mean V_3 tendency over a North Atlantic box. Swap in your own.

In [ ]:
mask = pg.box(35, 60, 280, 320)           # lat0, lat1, lon0, lon1 (0..360)
metric = lambda t: pg.box_mean(t[0, pg.ch("V_3")], mask)
which = ["L0H0", "L3H1", "L5H2", "L5H4", "L6MLP", "L7MLP"] if FAST else None   # None = all 56
sweep = pg.sweep_components(em, x, metric, which=which)
print("baseline:", round(sweep.attrs["baseline"], 3), "m/s per 6 h")
sweep.head(15).round(3)

In [ ]:
s = sweep.set_index("component").effect
order = pg.components(em)[1:]
s = s.reindex([c for c in order if c in s.index])
fig, ax = plt.subplots(figsize=(14, 3)); ax.bar(s.index, s.values); ax.tick_params(axis="x", rotation=90)
ax.set_ylabel("Δ metric"); ax.axhline(0, color="k", lw=0.5);

## Try this

1. Zero vs mean ablation: rerun the sweep with `mode="zero"`. Which components look important only under zero ablation, and why might that be misleading?
2. Ablate **all heads of one layer** (`heads=[(4, h) for h in range(6)]`). Which layer's attention is indispensable?
3. Ablate the most **upstream-looking** head from notebook 01. Does the error appear **downstream** (east) of strong gradients, as if advection were broken?
4. Ablation in a **rollout**: write a hook that ablates L5H2 at every step (`em.rollout(..., hooks_at=lambda t: {...})`) and see how the forecast degrades over 2 days.
5. This is the node-level start of circuit discovery. The `circuits` module and exege's `exege.circuits` do it at the **edge** level (Vi-CD). Pick your metric and compare.